# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [34]:
!git clone https://github.com/HasnainRaza16/flyrank-ml-Hasnain.git
%cd flyrank-ml-Hasnain

Cloning into 'flyrank-ml-Hasnain'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 122 (delta 36), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 1.85 MiB | 11.48 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/flyrank-ml-Hasnain/flyrank-ml-Hasnain/flyrank-ml-Hasnain/flyrank-ml-Hasnain


In [35]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.width', 200)

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
OUT_PATH = "work/outputs/baseline_action_score.csv"

df = pd.read_csv(DATA_PATH)
print("rows, cols:", df.shape)

# Outcome used ONLY for sanity-checking signals below, never as a rule input.
# Formula matches the data dictionary: is_declining_label = (trend_direction == "down")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print("base rate of decline label: %.3f  (n=%d)" % (df["is_declining_label"].mean(), len(df)))

rows, cols: (30000, 44)
base rate of decline label: 0.542  (n=30000)


## 1. Signal checks (before trusting anything)

**Signal A:staleness.** This is the signal behind FlyRank's *refresh flags*: a page that hasn't been touched in a long time is more likely to be slipping. I bucket by `freshness_tier` and compare decline rate per bucket against the base rate.

**Signal B:CTR vs. position.** This is the signal behind FlyRank's *CTR-fix logic*: better position should mean higher CTR, so a page whose CTR is far below what its position predicts is a real anomaly. I bucket by `position_tier` and compare median CTR.

Both are checked with n printed per bucket before any verdict.

In [36]:
order_fresh = ["0-30", "31-90", "91-180", "181+"]
sig_a = df.groupby("freshness_tier")["is_declining_label"].agg(mean_decline_rate="mean", n="count")
sig_a = sig_a.reindex(order_fresh)
print(sig_a)
print()
print("base rate for comparison: %.3f" % df["is_declining_label"].mean())

                mean_decline_rate      n
freshness_tier                          
0-30                     0.511377  20480
31-90                    0.588571    175
91-180                   0.611057   9171
181+                     0.471264    174

base rate for comparison: 0.542


Verdict on Signal A: MIXED

Decline rate does not rise cleanly with staleness. `0-30` (n=20,480) and `91-180` (n=9,171) — the two buckets with real sample size sit close to the base rate. The tail buckets (`31-90`, n=175; `181+`, n=174) are too small to trust, and `181+` actually shows the *lowest* decline rate, the opposite of what the refresh flag story predicts.

Read honestly: staleness alone is not a confirmed predictor of decline here. This is the kind of negative the assignment calls a win so it tells me not to lean on staleness by itself in the rule. I'll still use it as one necessary condition, but paired with the signal that does check out

In [37]:
order_pos = ["top_3", "page_1", "striking", "page_3_5", "deep", "no_data"]
sig_b = df.groupby("position_tier")["ctr"].agg(mean_ctr="mean", median_ctr="median", n="count")
sig_b = sig_b.reindex(order_pos)
print(sig_b)

               mean_ctr  median_ctr        n
position_tier                               
top_3          1.483611        0.00   2321.0
page_1         0.652467        0.16  11814.0
striking       0.323239        0.11   7304.0
page_3_5       0.222484        0.03   7242.0
deep           0.150212        0.00   1319.0
no_data             NaN         NaN      NaN


Median CTR falls in the expected order as position worsens: page_1 (n=11,814) highest, then striking (n=7,304), then page_3_5 (n=7,242), then deep (n=1,319)matching what CTR-fix logic assumes. `no_data` is excluded (no position data isn't a meaningful comparison).

Caveat worth flagging now: `top_3` (n=2,321) has median CTR of 0.00 because ~77% of those rows have zero recorded clicks a data quirk, not a modeling choice. This means the CTR-gap rule can never fire for `top_3` or `deep` pages (their tier median is already 0). Noted here so it isn't a silent gap later


## The rule, in plain words

"A page deserves a refresh review if it's stale, it's still getting meaningful search visibility, and its CTR is underperforming what pages in the same position bracket normally get."

Staleness alone didn't confirm (Signal A), so it never fires by itself,it's gated behind visibility and the CTR-position gap, which did confirm (Signal B).

- `stale` = `days_since_last_update >= 90`
- `visible` = `impressions_90d >= 500`
- `ctr_gap` = CTR below the median CTR for that row's `position_tier`

**Score** = `stale * visible * ctr_gap * impressions_90d`
**Reason code** = `stale_visible_ctr_gap` when the row qualifies, else `no_action`
**Action label** = `REFRESH_REVIEW` when the row qualifies, else `MONITOR`

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [38]:
# position-tier median CTR, computed on real position tiers only (excludes no_data)
tier_median_ctr = df.loc[df["position_tier"] != "no_data"].groupby("position_tier")["ctr"].median()

df["stale"] = (df["days_since_last_update"] >= 90).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["tier_median_ctr"] = df["position_tier"].map(tier_median_ctr).fillna(np.inf)
df["ctr_gap"] = ((df["position_tier"] != "no_data") & (df["ctr"] < df["tier_median_ctr"])).astype(int)

df["score"] = df["stale"] * df["visible"] * df["ctr_gap"] * df["impressions_90d"]
df["reason_code"] = np.where(df["score"] > 0, "stale_visible_ctr_gap", "no_action")
df["action"] = np.where(df["score"] > 0, "REFRESH_REVIEW", "MONITOR")

print("n stale:", df['stale'].sum())
print("n visible:", df['visible'].sum())
print("n ctr_gap:", df['ctr_gap'].sum())
print("n flagged for REFRESH_REVIEW:", (df['score'] > 0).sum(), "of", len(df))

n stale: 9345
n visible: 16726
n ctr_gap: 13079
n flagged for REFRESH_REVIEW: 2019 of 30000


In [39]:
queue_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "ctr", "avg_position", "position_tier",
    "days_since_last_update", "freshness_tier",
    "stale", "visible", "ctr_gap", "score", "reason_code", "action",
]
queue = df[queue_cols].sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
queue.to_csv(OUT_PATH, index=False)
print("wrote", OUT_PATH, "-", queue.shape[0], "rows")
queue.head(10)

wrote work/outputs/baseline_action_score.csv - 30000 rows


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,ctr,avg_position,position_tier,days_since_last_update,freshness_tier,stale,visible,ctr_gap,score,reason_code,action
0,content_5fe46e04994d,client_4e07408562,keyword article,informational,517715,741,0.14,4.2,page_1,104,91-180,1,1,1,517715,stale_visible_ctr_gap,REFRESH_REVIEW
1,content_36ff89c8214e,client_19581e27de,keyword article,informational,295097,154,0.05,7.3,page_1,104,91-180,1,1,1,295097,stale_visible_ctr_gap,REFRESH_REVIEW
2,content_c8e9d6ab9013,client_19581e27de,keyword article,informational,208678,0,0.00,9.7,page_1,104,91-180,1,1,1,208678,stale_visible_ctr_gap,REFRESH_REVIEW
3,content_a7427266c305,client_19581e27de,keyword article,informational,201111,219,0.11,5.7,page_1,104,91-180,1,1,1,201111,stale_visible_ctr_gap,REFRESH_REVIEW
4,content_91652435f57a,client_19581e27de,keyword article,commercial,159590,100,0.06,7.8,page_1,104,91-180,1,1,1,159590,stale_visible_ctr_gap,REFRESH_REVIEW
5,content_f42eb861c6dd,client_19581e27de,keyword article,transactional,152467,191,0.13,6.5,page_1,104,91-180,1,1,1,152467,stale_visible_ctr_gap,REFRESH_REVIEW
6,content_11fcfd65d94c,client_19581e27de,keyword article,commercial,149083,227,0.15,6.2,page_1,104,91-180,1,1,1,149083,stale_visible_ctr_gap,REFRESH_REVIEW
7,content_97a86caf3a3d,client_19581e27de,keyword article,transactional,147670,97,0.07,6.4,page_1,104,91-180,1,1,1,147670,stale_visible_ctr_gap,REFRESH_REVIEW
8,content_8b36799b7e44,client_6208ef0f77,keyword article,informational,141400,23,0.02,32.0,page_3_5,104,91-180,1,1,1,141400,stale_visible_ctr_gap,REFRESH_REVIEW
9,content_c1fe78bc4e37,client_19581e27de,keyword article,commercial,134055,43,0.03,7.5,page_1,104,91-180,1,1,1,134055,stale_visible_ctr_gap,REFRESH_REVIEW


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [40]:
top20 = queue.head(20)
top20

,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,ctr,avg_position,position_tier,days_since_last_update,freshness_tier,stale,visible,ctr_gap,score,reason_code,action
0,content_5fe46e04994d,client_4e07408562,keyword article,informational,517715,741,0.14,4.2,page_1,104,91-180,1,1,1,517715,stale_visible_ctr_gap,REFRESH_REVIEW
1,content_36ff89c8214e,client_19581e27de,keyword article,informational,295097,154,0.05,7.3,page_1,104,91-180,1,1,1,295097,stale_visible_ctr_gap,REFRESH_REVIEW
2,content_c8e9d6ab9013,client_19581e27de,keyword article,informational,208678,0,0.00,9.7,page_1,104,91-180,1,1,1,208678,stale_visible_ctr_gap,REFRESH_REVIEW
3,content_a7427266c305,client_19581e27de,keyword article,informational,201111,219,0.11,5.7,page_1,104,91-180,1,1,1,201111,stale_visible_ctr_gap,REFRESH_REVIEW
4,content_91652435f57a,client_19581e27de,keyword article,commercial,159590,100,0.06,7.8,page_1,104,91-180,1,1,1,159590,stale_visible_ctr_gap,REFRESH_REVIEW
5,content_f42eb861c6dd,client_19581e27de,keyword article,transactional,152467,191,0.13,6.5,page_1,104,91-180,1,1,1,152467,stale_visible_ctr_gap,REFRESH_REVIEW
6,content_11fcfd65d94c,client_19581e27de,keyword article,commercial,149083,227,0.15,6.2,page_1,104,91-180,1,1,1,149083,stale_visible_ctr_gap,REFRESH_REVIEW
7,content_97a86caf3a3d,client_19581e27de,keyword article,transactional,147670,97,0.07,6.4,page_1,104,91-180,1,1,1,147670,stale_visible_ctr_gap,REFRESH_REVIEW
8,content_8b36799b7e44,client_6208ef0f77,keyword article,informational,141400,23,0.02,32.0,page_3_5,104,91-180,1,1,1,141400,stale_visible_ctr_gap,REFRESH_REVIEW
9,content_c1fe78bc4e37,client_19581e27de,keyword article,commercial,134055,43,0.03,7.5,page_1,104,91-180,1,1,1,134055,stale_visible_ctr_gap,REFRESH_REVIEW


1. **`content_5fe46e04994d`** (client `4e07408562`) `REFRESH_REVIEW`. Why: 517,715 impressions at `page_1` (avg position 4.2) but CTR only 0.14%, well under the page_1 median, last updated 104 days ago. Would be wrong if: this keyword's SERP now has a featured snippet or AI overview stealing clicks a low CTR at a good position isn't always a content problem.
2. **`content_36ff89c8214e`** (client `19581e27de`) — `REFRESH_REVIEW`. Why: 295,097 impressions, page_1 (7.3), CTR 0.05%, 104 days stale. Would be wrong if: this page's title/meta was already changed in the last few days and hasn't shown up in these metrics yet.
3. **`content_c8e9d6ab9013`** (same client)  `REFRESH_REVIEW`. Why: 208,678 impressions, page_1 (9.7), CTR literally 0.00%, stale 104 days. Would be wrong if: `clicks_90d` here is a tracking outage rather than a genuine zero click reality.
4. **`content_a7427266c305`** (same client) — `REFRESH_REVIEW`. Why: 201,111 impressions, page_1 (5.7), CTR 0.11%. Would be wrong if: this is a branded navigational query where users don't need to click through.
5. **`content_91652435f57a`** (same client) `REFRESH_REVIEW`. Why: 159,590 impressions, page_1 (7.8), CTR 0.06%, commercial intent. Would be wrong if: this page recently got a pricing update that hasn't propagated to search yet.
6. **`content_f42eb861c6dd`** (same client) `REFRESH_REVIEW`. Why: 152,467 impressions, page_1 (6.5), CTR 0.13%, transactional intent. Would be wrong if: this is a seasonal page and the low CTR is a temporary, expected dip.
7. **`content_11fcfd65d94c`** (same client) `REFRESH_REVIEW`. Why: 149,083 impressions, page_1 (6.2), CTR 0.15%, real word_count of 3,372 not thin content, a title/snippet problem specifically. Would be wrong if: the real fix is technical (schema markup) rather than content refresh.
8. **`content_97a86caf3a3d`** (same client) `REFRESH_REVIEW`. Why: 147,670 impressions, page_1 (6.4), CTR 0.07%. Would be wrong if: a near-duplicate page on the same site is cannibalizing its clicks.
9. **`content_8b36799b7e44`** (client `6208ef0f77`) —`REFRESH_REVIEW`. Why: 141,400 impressions but weaker position (page_3_5, avg 32.0), CTR 0.02% — a position problem, not just a CTR-at-good-position problem. Would be wrong if: position volatility (swinging between 3 and 60) makes the 32.0 average misleading.
10. **`content_c1fe78bc4e37`** (client `19581e27de`) `REFRESH_REVIEW`. Why: 134,055 impressions, page_1 (7.5), CTR 0.03%. Would be wrong if: this is one more instance of the same client-wide issue as #2–#9, not an independent problem.
11. **`content_e752a4e03dd3`** (client `6208ef0f77`) `REFRESH_REVIEW`. Why: 130,892 impressions, page_3_5 (23.9), CTR 0.01% same pattern as #9. Would be wrong if: this client's site moved templates/domains around day 104, making the position an artifact of transition.
12. **`content_54baba704595`** (same client) `REFRESH_REVIEW`. Why: 130,617 impressions, page_3_5 but avg position 47.0 (near the deep boundary), CTR 0.01%. Would be wrong if: this page swings between very good and very bad positions and 47.0 is a misleading average.
13. **`content_124763d39ca5`** (same client) `REFRESH_REVIEW`. Why: 129,803 impressions, page_3_5 (33.2), CTR 0.01% fourth row from this client with an almost identical profile to #9, #11, #12. Would be wrong if: these four share one template and the real fix is one template change, not four refreshes.
14. **`content_4c76e9b13aea`** (client `19581e27de`) `REFRESH_REVIEW`. Why: 127,952 impressions, page_1 (7.4), CTR 0.07%, transactional intent. Would be wrong if: 0.07% is actually normal CTR for this vertical at this position.
15. **`content_45fb95832c96`** (same client) `REFRESH_REVIEW`. Why: 126,633 impressions, page_1 (7.6), CTR 0.12%. Would be wrong if: a recent promo already fixed the click-through gap and this data predates it.
16. **`content_b115f7c74779`** (same client) `REFRESH_REVIEW`. Why: 123,469 impressions, page_1 (8.0), CTR 0.03%. Would be wrong if: this page ranks for an ambiguous keyword where most searchers want a different intent entirely.
17. **`content_647e177596e9`** (same client) `REFRESH_REVIEW`. Why: 111,222 impressions, page_1 (7.4), CTR 0.09%. Would be wrong if: this is a near-duplicate of #16 from the same page family.
18. **`content_42d423551e2c`** (client `6208ef0f77`) `REFRESH_REVIEW`. Why: 106,652 impressions, page_1 (4.8)  a strong position for this otherwise page_3_5-heavy client, CTR 0.09%. Would be wrong if: this row shows the client's page_3_5 rows (#9, #11–13) are just normal variance, not systemic.
19. **`content_5d3dfb80a423`** (client `19581e27de`) `REFRESH_REVIEW`. Why: 97,235 impressions, page_1 (6.5), CTR 0.07%. Would be wrong if: this page was already scheduled for a redesign next sprint.
20. **`content_867dc005867d`** (same client) `REFRESH_REVIEW`. Why: 95,522 impressions, page_1 (6.8), CTR 0.10%, informational intent. Would be wrong if: this CTR is already at the ceiling for this query type no real headroom to gain.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

**Weak pick called out:** 19 of the top 20 rows belong to just two clients (`19581e27de` and `6208ef0f77`), and every single row shares `days_since_last_update == 104` exactly. That's not 20 independent editorial judgments, it looks like one client-wide event (a bulk publish or bulk re-tag 104 days ago) driving the ranking, not 20 genuinely distinct problem pages. As written, the rule has no per-client normalization, so a client with unusual scale or a shared update cadence can flood the top of the queue. **Fix for a later iteration:** rank within client first, or add a max-per-client cap to the top of the queue, so the list isn't dominated by one account's traffic volume.

**Second weak spot:** Signal B showed `top_3` and `deep` position tiers have a median CTR of 0.00 (a data quirk, ~77% zero-click rows at `top_3`). That means the `ctr_gap` condition can never fire for pages already in the best or worst position tiers.. The rule is structurally blind to CTR problems at the extremes. Anything genuinely wrong with a `top_3` or `deep` page won't get caught by this baseline.

**Leakage check:**
- Rule inputs used: `days_since_last_update`, `impressions_90d`, `ctr`, `position_tier`. All are current-window, non-label columns.
- `trend_direction` / `trend_pct` / `is_declining_label` were used **only** in section 1, to check whether staleness holds up as a signal never as an input to `stale`, `visible`, `ctr_gap`, `score`, `reason_code`, or `action`.
- No forward/future-window columns and no warehouse `fact_content_query_90d`-style overlapping windows are touched notebook only reads the 30k-row starter CSV.
- No client names, URLs, or raw identifiers beyond the pseudonymous `content_id` / `client_id` appear anywhere above.

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.